# Reference metrics and judge ratings

Spearman rank correlations between BERTScore F1 or ModernBERTić cosine similarity and the four judge ratings on 315 validation turns per model. The question is whether responses that are closer to the recorded reference also receive higher judge ratings.

The analysis reports correlations within each model, then pools models by rubric and across rubrics. Pooling changes the question: a pooled correlation can reflect differences between models as well as differences between responses within a model.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import Markdown, display

import story_utils as S

items = S.load_items()
METRICS = ["bertscore_f1", "modernbertic_cosine"]
KEYS = ["call_id", "turn_id", "model"]

metrics = S.build_df_metrics(items)[KEYS + METRICS]
ratings = S.build_df_rubrics(items).dropna(subset=["score"])
ratings = ratings[ratings["model"] != S.GT]
long = ratings.merge(metrics, on=KEYS, validate="many_to_one")
MODELS = [model for model in S.ALL_ORDER if model in set(long["model"])]

mean_ratings = (
    long.groupby(KEYS, observed=True)
    .agg(score=("score", "mean"),
         bertscore_f1=("bertscore_f1", "first"),
         modernbertic_cosine=("modernbertic_cosine", "first"))
    .reset_index()
)

## Calculation

Ratings are ordinal (1–3), so the analysis uses rank correlation. Missing metric–rating pairs are dropped separately for each calculation. Correlations are undefined when either variable is constant or fewer than three pairs remain.

The reported p-values are the nominal values returned by SciPy. Turns from the same conversation and ratings of the same turn are dependent; pooled rows are not independent observations. These p-values are not adjusted for that dependence or for multiple comparisons and should not be treated as confirmatory evidence.

In [2]:
def correlation(frame, metric):
    paired = frame.dropna(subset=[metric, "score"])
    if len(paired) < 3 or paired[metric].nunique() < 2 or paired["score"].nunique() < 2:
        return {"Spearman rho": np.nan, "Nominal p-value": np.nan}
    rho, p_value = spearmanr(paired[metric], paired["score"])
    return {"Spearman rho": rho, "Nominal p-value": p_value}


FORMAT = {"Spearman rho": "{:+.3f}", "Nominal p-value": "{:.1e}"}

## Within each model

Each coefficient compares one metric with one rubric across the validation turns for a single model. This is the most direct view of whether a metric distinguishes higher- and lower-rated responses from that model.

In [3]:
for metric in METRICS:
    rows = [
        {"Model": S.model_label(model), "Rubric": S.RUBRIC_LABELS[rubric],
         **correlation(long[(long["model"] == model) & (long["rubric"] == rubric)], metric)}
        for model in MODELS for rubric in S.RUBRIC_ORDER
    ]
    within_model = pd.DataFrame(rows).set_index(["Model", "Rubric"])
    display(Markdown(f"### {S.METRIC_LABELS[metric]}"))
    display(within_model.style.format(FORMAT).background_gradient(
        cmap="RdBu_r", subset=["Spearman rho"], vmin=-1, vmax=1,
    ))

### BERTScore F1

### ModernBERTić cosine

## Pooled by rubric

Each coefficient uses all model–turn pairs for one rubric. Compare these values with the within-model results before interpreting a pooled association.

In [4]:
pooled_by_rubric = pd.DataFrame([
    {"Metric": S.METRIC_LABELS[metric], "Rubric": S.RUBRIC_LABELS[rubric],
     **correlation(long[long["rubric"] == rubric], metric)}
    for metric in METRICS for rubric in S.RUBRIC_ORDER
]).set_index(["Metric", "Rubric"])

pooled_by_rubric.style.format(FORMAT).background_gradient(
    cmap="RdBu_r", subset=["Spearman rho"], vmin=-1, vmax=1,
)

## Pooled across models and rubrics

The first calculation includes every model–turn–rubric row, repeating each metric value for the four rubrics. The second uses the mean rubric score for each model–turn pair. Both are descriptive summaries; neither establishes that a reference metric can replace the rubric judge.

In [5]:
pooled = pd.DataFrame([
    {"Metric": S.METRIC_LABELS[metric], "Aggregation": label,
     **correlation(frame, metric)}
    for metric in METRICS
    for label, frame in [("All model–turn–rubric rows", long),
                         ("Mean rubric score per model–turn", mean_ratings)]
]).set_index(["Metric", "Aggregation"])

pooled.style.format(FORMAT).background_gradient(
    cmap="RdBu_r", subset=["Spearman rho"], vmin=-1, vmax=1,
)

## Reading the results

A positive coefficient means higher reference similarity tends to accompany higher judge ratings. A weak coefficient means the rankings have little monotonic agreement; it does not show which measure is correct. Differences between pooled and within-model coefficients can arise from model-level differences.

These correlations concern this validation set and this judge. They do not measure agreement with clinicians. The separate response analyses provide examples of where the two kinds of score disagree.